In [ ]:
import asyncio
import json
import os
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")


## Korak 1: Definirajte Pydantic modele za strukturirane izlaze

Ovi modeli definiraju **šemu** koju će agenti vraćati. Korištenje `response_format` s Pydantic-om osigurava:
- ✅ Sigurnu ekstrakciju podataka tipa
- ✅ Automatsku validaciju
- ✅ Nema pogrešaka prilikom parsiranja iz slobodnog teksta
- ✅ Jednostavno uvjetno usmjeravanje na temelju polja


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## Korak 2: Izradite alat za rezervaciju hotela

Ovaj alat je ono što će **availability_agent** pozvati da provjeri jesu li sobe dostupne. Koristimo dekorator `@ai_function` za:
- Pretvaranje Python funkcije u AI-pozivni alat
- Automatsko generiranje JSON sheme za LLM
- Obradu provjere parametara
- Omogućavanje automatskog poziva od strane agenata

Za ovaj demo:
- **Stockholm, Seattle, Tokyo, London, Amsterdam** → Ima soba ✅
- **Sve ostale gradove** → Nema soba ❌


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## Korak 3: Definirajte funkcije uvjeta za usmjeravanje

Ove funkcije pregledavaju odgovor agenta i određuju koji put slijediti u tijeku rada.

**Ključni obrazac:**
1. Provjerite je li poruka `AgentExecutorResponse`
2. Parsirajte strukturirani izlaz (Pydantic model)
3. Vratite `True` ili `False` za kontrolu usmjeravanja

Tijek rada će procijeniti ove uvjete na **granicama** da odluči kojeg izvršitelja pozvati sljedećeg.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels ARE available.
    
    Returns True if the destination has hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return True  # Default to True if unexpected type

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels are NOT available.
    
    Returns True if the destination has no hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception as e:
        return False


print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")

## Korak 4: Stvaranje prilagođenog izvršitelja prikaza

Izvršitelji su komponente tijeka rada koje obavljaju transformacije ili sporedne učinke. Koristimo dekorator `@executor` za stvaranje prilagođenog izvršitelja koji prikazuje konačni rezultat.

**Ključni pojmovi:**
- `@executor(id="...")` - Registrira funkciju kao izvršitelja tijeka rada
- `WorkflowContext[Never, str]` - Tipične naznake za ulaz/izlaz
- `ctx.yield_output(...)` - Generira konačni rezultat tijeka rada


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created with @executor decorator")

## Korak 5: Učitajte Varijable Okruženja

Konfigurirajte LLM klijenta. Ovaj primjer radi sa:
- **Microsoft Foundry** / **Azure OpenAI** (Responses API — preporučeno)
- **Azure OpenAI**
- **OpenAI**


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Microsoft Foundry provider with keyless authentication
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


## Korak 6: Kreirajte AI agente sa strukturiranim izlazima

Kreiramo **tri specijalizirana agenta**, svaki omotan u `AgentExecutor`:

1. **availability_agent** - Provjerava dostupnost hotela koristeći alat
2. **alternative_agent** - Predlaže alternativne gradove (kada nema soba)
3. **booking_agent** - Poticanje rezervacije (kada su sobe dostupne)

**Ključne značajke:**
- `tools=[hotel_booking]` - Agentu se pruža alat
- `response_format=PydanticModel` - Prisiljava strukturu JSON izlaza
- `AgentExecutor(..., id="...")` - Omotava agenta za korištenje u tijeku rada


In [ ]:
# Agent 1: Check availability with tool
availability_agent = AgentExecutor(
    provider.as_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    provider.as_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    provider.as_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)


## Korak 7: Izgradnja tijeka rada s uvjetnim bridovima

Sada koristimo `WorkflowBuilder` za konstrukciju grafa s uvjetnim usmjeravanjem:

**Struktura tijeka rada:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙         ↘
[no_availability]  [has_availability]
        ↓              ↓
alternative_agent  booking_agent
        ↓              ↓
    display_result ←───┘
```

**Ključne metode:**
- `.set_start_executor(...)` - Postavlja ulaznu točku
- `.add_edge(from, to, condition=...)` - Dodaje uvjetni brid
- `.build()` - Završava tijek rada


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing:</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result
        </p>
    </div>
""")
)

## Korak 8: Pokreni Testni Slučaj 1 - Grad BEZ raspoloživosti (Pariz)

Testirajmo put bez **raspoloživosti** tražeći hotele u Parizu (koji nema soba u našoj simulaciji).


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → alternative_agent → display_result</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run the workflow
events_paris = await workflow.run(request_paris)
outputs_paris = events_paris.get_outputs()

# Display results
if outputs_paris:
    result_paris = AlternativeResult.model_validate_json(outputs_paris[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT (Paris)</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_paris.alternative_destination}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_paris.reason}</p>
            </div>
        </div>
    """)
    )

## Korak 9: Pokreni Test Slučaj 2 - Grad S DOSTUPNOŠĆU (Stockholm)

Sada testirajmo put dostupnosti **availability** tako da zatražimo hotele u Stockholmu (koji ima sobe u našoj simulaciji).


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run the workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
            </div>
        </div>
    """)
    )

## Glavne pouke i sljedeći koraci

### ✅ Što ste naučili:

1. **WorkflowBuilder obrazac**
   - Koristite `.set_start_executor()` za definiranje ulazne točke
   - Koristite `.add_edge(from, to, condition=...)` za uvjetno usmjeravanje
   - Pozovite `.build()` za finalizaciju tijeka rada

2. **Uvjetno usmjeravanje**
   - Funkcije uvjeta pregledavaju `AgentExecutorResponse`
   - Parsiraju strukturirane izlaze za donošenje odluka o usmjeravanju
   - Vraćaju `True` za aktivaciju grane, `False` za preskakanje

3. **Integracija alata**
   - Koristite `@ai_function` za pretvaranje Python funkcija u AI alate
   - Agent automatski poziva alate kad su potrebni
   - Alati vraćaju JSON koji agenti mogu parsirati

4. **Strukturirani izlazi**
   - Koristite Pydantic modele za sigurno izdvajanje podataka
   - Postavite `response_format=MyModel` prilikom kreiranja agenata
   - Parsirajte odgovore sa `Model.model_validate_json()`

5. **Prilagođeni izvršitelji**
   - Koristite `@executor(id="...")` za stvaranje komponenti tijeka rada
   - Izvršitelji mogu transformirati podatke ili izvoditi sporedne efekte
   - Koristite `ctx.yield_output()` za proizvodnju rezultata tijeka rada

### 🚀 Primjene u stvarnom svijetu:

- **Rezervacija putovanja**: Provjera dostupnosti, predlaganje alternativa, usporedba opcija
- **Korisnička podrška**: Usmjeravanje ovisno o tipu problema, raspoloženju, prioritetu
- **E-trgovina**: Provjera zaliha, predlaganje alternativa, obrada narudžbi
- **Moderacija sadržaja**: Usmjeravanje ovisno o toksičnosti, prijavama korisnika
- **Tijekovi odobrenja**: Usmjeravanje ovisno o iznosu, ulozi korisnika, razini rizika
- **Višestupanjska obrada**: Usmjeravanje ovisno o kvaliteti i potpunosti podataka

### 📚 Sljedeći koraci:

- Dodajte složenije uvjete (više kriterija)
- Implementirajte petlje s upravljanjem stanja tijeka rada
- Dodajte podtijekove rada za višekratno korištenje komponenti
- Integrirajte s pravim API-jima (rezervacija hotela, sustavi zaliha)
- Dodajte rukovanje pogreškama i rezervne puteve
- Vizualizirajte tijekove rada pomoću ugrađenih alata za vizualizaciju


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Napomena**:
Ovaj dokument je preveden korištenjem AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati greške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporuča se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazumevanja ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
